In [1]:
import sys
import pickle
import random
import copy
from typing import Optional, Union

from poke_env import RandomPlayer
from poke_env.data import GenData
from poke_env import AccountConfiguration
from poke_env.environment import SinglesEnv, SingleAgentWrapper
from poke_env.environment.env import _EnvPlayer
from poke_env.battle import AbstractBattle, Battle
from poke_env.teambuilder import Teambuilder
from poke_env.player import Player, RandomPlayer



import numpy as np
import numpy.typing as npt
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

from collections import defaultdict
from typing import Any, Dict, Optional

from gymnasium.spaces import Box, Discrete, Space, MultiDiscrete
from gymnasium.spaces import Dict as gymdict
import torch
import torch.nn as nn
from torch import multiprocessing
from tensordict import TensorDict, TensorDictBase
from tensordict.nn import TensorDictModule
from tensordict.nn.distributions import NormalParamExtractor


from torchrl.collectors import SyncDataCollector
from torchrl.data.replay_buffers import ReplayBuffer
from torchrl.data.replay_buffers.samplers import SamplerWithoutReplacement
from torchrl.data.replay_buffers.storages import LazyTensorStorage
from torchrl.envs import (Compose, DoubleToFloat, ObservationNorm, StepCounter,
                          TransformedEnv)


#Custom Env pytorch tutorial
from typing import Optional
from torchrl.data import BoundedTensorSpec, CompositeSpec, UnboundedContinuousTensorSpec
from torchrl.envs import (
    CatTensors,
    EnvBase,
    Transform,
    TransformedEnv,
    UnsqueezeTransform,
)
from torchrl.envs.transforms.transforms import _apply_to_composite
from torchrl.envs.utils import step_mdp
#---------------------------
from torchrl.envs.libs.gym import GymEnv
from torchrl.envs.utils import check_env_specs, ExplorationType, set_exploration_type
from torchrl.modules import ProbabilisticActor, TanhNormal, ValueOperator
from torchrl.objectives import ClipPPOLoss
from torchrl.objectives.value import GAE
from tqdm import tqdm

from ray import shutdown
from ray.rllib.algorithms import PPOConfig
from ray.rllib.core import Columns
from ray.rllib.core.rl_module import RLModuleSpec
from ray.rllib.core.rl_module.apis.value_function_api import ValueFunctionAPI
from ray.rllib.core.rl_module.torch import TorchRLModule
from ray.rllib.env import ParallelPettingZooEnv
from ray.tune.registry import register_env

In [2]:
"""
cd "C:\Austin\Self_Projects\Pokemon_Sim\pokemon-showdown"
node pokemon-showdown start --no-security
"""

'\ncd "C:\\Austin\\Self_Projects\\Pokemon_Sim\\pokemon-showdown"\nnode pokemon-showdown start --no-security\n'

In [3]:
#Hyperparams 
format="gen9ou"
device="cpu"

In [4]:
team1 = """
Meruem (Kingambit) @ Leftovers  
Ability: Supreme Overlord  
Tera Type: Ghost  
EVs: 164 HP / 252 Atk / 92 Spe  
Adamant Nature  
- Sucker Punch  
- Iron Head  
- Kowtow Cleave  
- Swords Dance  

Moebius (Deoxys-Speed) @ Life Orb  
Ability: Pressure  
Tera Type: Psychic  
EVs: 28 HP / 252 SpA / 228 Spe  
Modest Nature  
IVs: 0 Atk  
- Nasty Plot  
- Focus Blast  
- Psycho Boost  
- Shadow Ball  

Anomaly (Great Tusk) @ Booster Energy  
Ability: Protosynthesis  
Tera Type: Steel  
EVs: 252 Atk / 4 Def / 252 Spe  
Jolly Nature  
- Headlong Rush  
- Ice Spinner  
- Rapid Spin  
- Close Combat  

Yoshi (Dragonite) @ Choice Band  
Ability: Multiscale  
Shiny: Yes  
Tera Type: Normal  
EVs: 16 HP / 252 Atk / 240 Spe  
Adamant Nature  
- Outrage  
- Extreme Speed  
- Ice Spinner  
- Fire Punch  

Anomаly (Iron Moth) @ Booster Energy  
Ability: Quark Drive  
Tera Type: Ground  
EVs: 124 Def / 132 SpA / 252 Spe  
Timid Nature  
- Fiery Dance  
- Sludge Wave  
- Tera Blast  
- Toxic Spikes  

Siren (Primarina) @ Assault Vest  
Ability: Torrent  
Tera Type: Poison  
EVs: 76 HP / 252 SpA / 180 Spe  
Modest Nature  
IVs: 0 Atk  
- Surf  
- Moonblast  
- Whirlpool  
- Psychic Noise  
"""

In [5]:
team2 = """
Torkoal @ Heat Rock  
Ability: Drought  
Tera Type: Ground  
EVs: 104 HP / 252 SpA / 152 SpD  
Quiet Nature  
- Eruption  
- Overheat  
- Earthquake  
- Stealth Rock  

Hatterene @ Air Balloon  
Ability: Magic Bounce  
Tera Type: Ghost  
EVs: 252 HP / 252 Def / 4 SpD  
Relaxed Nature  
IVs: 0 Atk / 0 Spe  
- Trick Room  
- Psychic Noise  
- Dazzling Gleam  
- Healing Wish  

Raging Bolt @ Life Orb  
Ability: Protosynthesis  
Tera Type: Ghost  
EVs: 36 Def / 252 SpA / 220 Spe  
Modest Nature  
IVs: 20 Atk  
- Thunderclap  
- Weather Ball  
- Dragon Pulse  
- Solar Beam  

Slither Wing @ Assault Vest  
Ability: Protosynthesis  
Tera Type: Fire  
EVs: 40 HP / 252 Atk / 216 Spe  
Adamant Nature  
- U-turn  
- First Impression  
- Earthquake  
- Temper Flare  

Venusaur @ Life Orb  
Ability: Chlorophyll  
Tera Type: Fire  
EVs: 4 Atk / 252 SpA / 252 Spe  
Naive Nature  
- Growth  
- Giga Drain  
- Weather Ball  
- Earthquake  

Walking Wake @ Wise Glasses  
Ability: Protosynthesis  
Tera Type: Fairy  
EVs: 12 HP / 244 SpA / 252 Spe  
Timid Nature  
- Hydro Steam  
- Weather Ball  
- Draco Meteor  
- Flip Turn  

"""

In [6]:
class SmogonEnv(SinglesEnv):

    def __init__(
        self,
        #SinglesEnv Variables
        account_configuration1: Optional[AccountConfiguration] = None,
        account_configuration2: Optional[AccountConfiguration] = None,
        battle_format: str = "gen9ou",
        start_timer_on_battle_start: bool = True,
        strict = True,
        fake = False,

        #Custom Input
        team1: Optional[Union[str, Teambuilder]] = None,
        team2: Optional[Union[str, Teambuilder]] = None,
    ):

        SinglesEnv.__init__(
            self,
            #DoublesEnv Variables
            account_configuration1=account_configuration1,
            account_configuration2=account_configuration2,
            battle_format=battle_format,
            start_timer_on_battle_start=start_timer_on_battle_start,
            strict=strict,
            fake=fake,
            log_level=25
        )
        self.agent1.teampreview = self.teampreview
        self.agent2.teampreview = self.teampreview

        self.agent1.update_team(team1)
        self.agent2.update_team(team2)

        self.obs_low = np.array([-1, -1, -1, -1, 0, 0, 0, 0, 0, 0])
        self.obs_high = np.array([3, 3, 3, 3, 4, 4, 4, 4, 1, 1])
        
        self.observation_spaces = {
            agent: gymdict(
                {
                    "embeddings":Box(
                            low=self.obs_low,
                            high=self.obs_high,
                            dtype=np.float64,
                        ),
                    "valid_action_mask":Box(
                            low=np.zeros(10),
                            high=np.ones(10),
                            dtype=np.float64,
                        )
                }
            )
            for agent in self.possible_agents
        }

    #SinglesEnv Required Functions:

    def calc_reward(self, battle) -> float:
        #Initially using built in reward helper function provided by Poke-Env
        #For simplicity of initiall implementation and testing

        return self.reward_computing_helper(
            battle, fainted_value=2.0, hp_value=1.0, victory_value=30.0
        )


    def embed_battle(self, battle: Battle): #Will write a custom reward function as required - __NEED TO UPDATE FOR DOUBLE BATTLE__
        assert isinstance(battle, Battle)
        # -1 indicates that the move does not have a base power
        # or is not available
        moves_base_power = -np.ones(4)
        moves_dmg_multiplier = np.ones(4)
        for i, move in enumerate(battle.available_moves):
            moves_base_power[i] = (
                move.base_power / 100
            )  # Simple rescaling to facilitate learning
            if battle.opponent_active_pokemon is not None:
                moves_dmg_multiplier[i] = move.type.damage_multiplier(
                    battle.opponent_active_pokemon.type_1,
                    battle.opponent_active_pokemon.type_2,
                    type_chart=battle.opponent_active_pokemon._data.type_chart,
                )

        # We count how many pokemons have fainted in each team
        fainted_mon_team = len([mon for mon in battle.team.values() if mon.fainted]) / 6
        fainted_mon_opponent = (
            len([mon for mon in battle.opponent_team.values() if mon.fainted]) / 6
        )

        # Valid Action Mask
        valid_action_mask = np.ones(10)
        active_mon = battle.active_pokemon
        valid_action_mask[6:] = np.array([(move in battle.available_moves) for move in active_mon.moves.values()],dtype=np.float64)
        valid_action_mask[:6] = np.array([(mon in battle.available_switches) for mon in battle.team.values()],dtype=np.float64)



        # Final vector with 10 components
        final_vector = np.concatenate(
            [
                moves_base_power,
                moves_dmg_multiplier,
                [fainted_mon_team, fainted_mon_opponent],
            ]
        )
        return {"embeddings":torch.Tensor(final_vector), "valid_action_mask":valid_action_mask}
        
    
    def teampreview(self, battle: Battle) -> str: #Will write a custom reward function as required
        members = [1,2,3,4,5,6]#list(range(1, 7))
        random.shuffle(members)
        team_string = "/team " + "".join([str(x) for x in members])
        return team_string

    #Helper Functions:

    def print_teams(self):
        print(self.agent1._team.yield_team())
        print(self.agent2._team.yield_team())

    def print_torchrl_env_stats(self):
        print(self.device)
        print(self.batch_size)

    def select_game_team(self):
        pass

    def set_battle(self):
        self.battle1._finished=False
        self.battle2._finished=False

In [7]:
def create_multi_agent_env(env_config: Dict[str, Any]) -> ParallelPettingZooEnv:
    
    env = SmogonEnv(battle_format=env_config["battle_format"], strict=True, fake = True, team1=env_config["team1"], team2=env_config["team2"], start_timer_on_battle_start=True)

    return ParallelPettingZooEnv(env)


In [8]:
import random
from typing import List, Optional

from poke_env.battle.abstract_battle import AbstractBattle
from poke_env.battle.double_battle import DoubleBattle
from poke_env.battle.move_category import MoveCategory
from poke_env.battle.pokemon import Pokemon
from poke_env.battle.side_condition import SideCondition
from poke_env.battle.target import Target
from poke_env.player.battle_order import (
    BattleOrder,
    DefaultBattleOrder,
    DoubleBattleOrder,
    ForfeitBattleOrder,
    SingleBattleOrder,
)
from poke_env.player.player import Player

class Valid_Random_Player(Player):
    def choose_move(self, battle: AbstractBattle) -> BattleOrder:
        valid_action_mask = np.ones(10)
        action_list = [None, None, None, None, None, None, None, None, None, None]
        active_mon = battle.active_pokemon
        
        valid_action_mask[6:] = np.array([(move in battle.available_moves) for move in active_mon.moves.values()],dtype=np.float64)
        valid_action_mask[:6] = np.array([(mon in battle.available_switches) for mon in battle.team.values()],dtype=np.float64)

        action_list[:6] = [mon for mon in battle.team.values()]
        action_list[6:] = [move for move in active_mon.moves.values()]
        #print(valid_action_mask)

        action = np.where(valid_action_mask==1.0)[0]
        print(action)
        if len(action) != 0:
            action = np.random.choice(action)
            return self.create_order(action_list[action])
        else:
            action_list = [move for move in active_mon.moves.values()]
            return self.create_order(action_list[-1])

In [9]:
def create_single_agent_env(env_config: Dict[str, Any]) -> SingleAgentWrapper:
    
    env = SmogonEnv(battle_format=env_config["battle_format"], strict=True, fake = False, team1=env_config["team1"], team2=env_config["team2"], start_timer_on_battle_start=True)
    r = np.random.randint(10000)
    r = AccountConfiguration("acc"+str(r),"")
    opponent = Valid_Random_Player(account_configuration=r)
    return SingleAgentWrapper(env, opponent)

## Train Loop Test

In [10]:
register_env("showdown", create_multi_agent_env)
config = PPOConfig()

In [11]:
obs_low = np.array([-1, -1, -1, -1, 0, 0, 0, 0, 0, 0])
obs_high = np.array([3, 3, 3, 3, 4, 4, 4, 4, 1, 1])

In [12]:
config = config.environment(
    "showdown",
    env_config={"battle_format": format,
                "team1": team1,
                "team2": team2,
                },
    disable_env_checking=True,
)

In [13]:
class ActorCriticModule(TorchRLModule, ValueFunctionAPI):
    def __init__(
        self,
        observation_space: Space,
        action_space: Space,
        inference_only: bool,
        model_config: Dict[str, Any],
        catalog_class: Any,
    ):
        super().__init__(
            observation_space=observation_space,
            action_space=action_space,
            inference_only=inference_only,
            model_config=model_config,
            catalog_class=catalog_class,
        )
        self.model = nn.Linear(10, 100)
        self.actor = nn.Linear(100, 10)
        self.critic = nn.Linear(100, 1)
        self.val_test_sf = nn.Softmax()

    def _forward(self, batch: Dict[str, Any], **kwargs) -> Dict[str, Any]:
        obs = batch[Columns.OBS]["embeddings"]
        valid_act_map = batch[Columns.OBS]["valid_action_mask"]
        embeddings = self.model(obs)
        logits = self.actor(embeddings)

        action = self.val_test_sf(logits[0])
        action = action.argmax()
        
        #print(valid_act_map)
        #print(valid_act_map[0][action] == 0)
        if valid_act_map[0][action] == 0:
            logits = valid_act_map
            
            print("map being changed")

        return {Columns.EMBEDDINGS: embeddings, Columns.ACTION_DIST_INPUTS: logits}

    def compute_values(
        self, batch: Dict[str, Any], embeddings: Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        if embeddings is None:
            embeddings = self.model(batch[Columns.OBS]["embeddings"])
        return self.critic(embeddings).squeeze(-1)
    
    

In [ ]:
config = config.learners(num_learners=1)
config = config.env_runners(num_env_runners=1, explore=False)

config = config.multi_agent(
    policies={"p1"},
    policy_mapping_fn=lambda agent_id, ep_type: "p1",
    policies_to_train=["p1"],
)

config = config.rl_module(
    rl_module_spec=RLModuleSpec(
        module_class=ActorCriticModule,
        observation_space=gymdict(
                {
                    "embeddings":Box(
                            low=obs_low,
                            high=obs_high,
                            dtype=np.float64,
                        ),
                    "valid_action_mask":Box(
                            low=np.zeros(10),
                            high=np.ones(10),
                            dtype=np.float64,
                        )
                }
            ),
        action_space=Discrete(10),
        model_config={},
    )
)
config = config.training(
    gamma=0.99, lr=1e-3, train_batch_size=1024, num_epochs=100, minibatch_size=64
)

In [15]:
shutdown()

In [16]:
import os
os.environ["RAY_DEDUP_LOGS"] = "0"
algo = config.build_algo()

2025-11-20 20:13:52,082	WARNING algorithm_config.py:5033 -- You are running PPO on the new API stack! This is the new default behavior for this algorithm. If you don't want to use the new API stack, set `config.api_stack(enable_rl_module_and_learner=False,enable_env_runner_and_connector_v2=False)`. For a detailed migration guide, see here: https://docs.ray.io/en/master/rllib/new-api-stack-migration-guide.html
c:\Users\anaconda3\envs\poke_regj_env\lib\site-packages\ray\rllib\algorithms\algorithm.py:520: RayDeprecationWarning: This API is deprecated and may be removed in future Ray releases. You could suppress this warning by setting env variable PYTHONWARNINGS="ignore::DeprecationWarning"
`UnifiedLogger` will be removed in Ray 2.7.
  return UnifiedLogger(config, logdir, loggers=None)
c:\Users\anaconda3\envs\poke_regj_env\lib\site-packages\ray\tune\logger\unified.py:53: RayDeprecationWarning: This API is deprecated and may be removed in future Ray releases. You could suppress this warnin

(MultiAgentEnvRunner pid=15752) 2025-11-20 20:14:12,509 - SmogonEnv cqo0r - PS_ERROR - Error message received: |error|[Invalid choice] Can't do anything: The game is over
(MultiAgentEnvRunner pid=15752) 2025-11-20 20:14:12,510 - SmogonEnv cqo0r - CRITICAL - Unexpected error message: ['', 'error', "[Invalid choice] Can't do anything: The game is over"]
(MultiAgentEnvRunner pid=15752) c:\Users\anaconda3\envs\poke_regj_env\lib\site-packages\torch\nn\modules\module.py:1751: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
(MultiAgentEnvRunner pid=15752)   return self._call_impl(*args, **kwargs)


(MultiAgentEnvRunner pid=15752) {'SmogonEnv cqo0r': np.int32(1), 'SmogonEnv asud0': np.int32(3)}
(MultiAgentEnvRunner pid=15752) {'SmogonEnv cqo0r': np.int32(3), 'SmogonEnv asud0': np.int32(1)}
(MultiAgentEnvRunner pid=15752) {'SmogonEnv cqo0r': np.int32(1), 'SmogonEnv asud0': np.int32(3)}
(MultiAgentEnvRunner pid=15752) {'SmogonEnv cqo0r': np.int32(3), 'SmogonEnv asud0': np.int32(1)}
(MultiAgentEnvRunner pid=15752) {'SmogonEnv cqo0r': np.int32(1), 'SmogonEnv asud0': np.int32(3)}
(MultiAgentEnvRunner pid=15752) {'SmogonEnv cqo0r': np.int32(3), 'SmogonEnv asud0': np.int32(1)}
(MultiAgentEnvRunner pid=15752) {'SmogonEnv cqo0r': np.int32(1), 'SmogonEnv asud0': np.int32(3)}
(MultiAgentEnvRunner pid=15752) {'SmogonEnv cqo0r': np.int32(3), 'SmogonEnv asud0': np.int32(1)}
(MultiAgentEnvRunner pid=15752) {'SmogonEnv cqo0r': np.int32(1), 'SmogonEnv asud0': np.int32(3)}
(MultiAgentEnvRunner pid=15752) {'SmogonEnv cqo0r': np.int32(3), 'SmogonEnv asud0': np.int32(1)}
(MultiAgentEnvRunner pid=15752

(MultiAgentEnvRunner pid=15752) 2025-11-20 20:14:15,920 - SmogonEnv asud0 - WARNING - Received 'bigerror' message: ['', 'bigerror', "You will auto-tie if the battle doesn't end in 500 turns (on turn 1000)."]
(MultiAgentEnvRunner pid=15752) 2025-11-20 20:14:15,920 - SmogonEnv cqo0r - WARNING - Received 'bigerror' message: ['', 'bigerror', "You will auto-tie if the battle doesn't end in 500 turns (on turn 1000)."]


(MultiAgentEnvRunner pid=15752) {'SmogonEnv cqo0r': np.int32(3), 'SmogonEnv asud0': np.int32(1)}
(MultiAgentEnvRunner pid=15752) {'SmogonEnv cqo0r': np.int32(1), 'SmogonEnv asud0': np.int32(3)}
(MultiAgentEnvRunner pid=15752) {'SmogonEnv cqo0r': np.int32(3), 'SmogonEnv asud0': np.int32(1)}
(MultiAgentEnvRunner pid=15752) {'SmogonEnv cqo0r': np.int32(1), 'SmogonEnv asud0': np.int32(3)}
(MultiAgentEnvRunner pid=15752) {'SmogonEnv cqo0r': np.int32(3), 'SmogonEnv asud0': np.int32(1)}
(MultiAgentEnvRunner pid=15752) {'SmogonEnv cqo0r': np.int32(1), 'SmogonEnv asud0': np.int32(3)}
(MultiAgentEnvRunner pid=15752) {'SmogonEnv cqo0r': np.int32(3), 'SmogonEnv asud0': np.int32(1)}
(MultiAgentEnvRunner pid=15752) {'SmogonEnv cqo0r': np.int32(1), 'SmogonEnv asud0': np.int32(3)}
(MultiAgentEnvRunner pid=15752) {'SmogonEnv cqo0r': np.int32(3), 'SmogonEnv asud0': np.int32(1)}
(MultiAgentEnvRunner pid=15752) {'SmogonEnv cqo0r': np.int32(1), 'SmogonEnv asud0': np.int32(3)}
(MultiAgentEnvRunner pid=15752

...

(MultiAgentEnvRunner pid=15752) {'SmogonEnv cqo0r': np.int32(3), 'SmogonEnv asud0': np.int32(1)}
(MultiAgentEnvRunner pid=15752) {'SmogonEnv cqo0r': np.int32(1), 'SmogonEnv asud0': np.int32(3)}
(MultiAgentEnvRunner pid=15752) {'SmogonEnv cqo0r': np.int32(3), 'SmogonEnv asud0': np.int32(1)}
(MultiAgentEnvRunner pid=15752) {'SmogonEnv cqo0r': np.int32(1), 'SmogonEnv asud0': np.int32(3)}
(MultiAgentEnvRunner pid=15752) {'SmogonEnv cqo0r': np.int32(3), 'SmogonEnv asud0': np.int32(1)}
(MultiAgentEnvRunner pid=15752) {'SmogonEnv cqo0r': np.int32(1), 'SmogonEnv asud0': np.int32(3)}
(MultiAgentEnvRunner pid=15752) {'SmogonEnv cqo0r': np.int32(3), 'SmogonEnv asud0': np.int32(1)}
(MultiAgentEnvRunner pid=15752) {'SmogonEnv cqo0r': np.int32(1), 'SmogonEnv asud0': np.int32(3)}
(MultiAgentEnvRunner pid=15752) {'SmogonEnv cqo0r': np.int32(3), 'SmogonEnv asud0': np.int32(1)}
(MultiAgentEnvRunner pid=15752) {'SmogonEnv cqo0r': np.int32(1), 'SmogonEnv asud0': np.int32(3)}
(MultiAgentEnvRunner pid=15752

(MultiAgentEnvRunner pid=15752) 2025-11-20 20:16:39,231 - SmogonEnv asud0 - WARNING - Received 'bigerror' message: ['', 'bigerror', "You will auto-tie if the battle doesn't end in 30 turns (on turn 1000)."]
(MultiAgentEnvRunner pid=15752) 2025-11-20 20:16:39,231 - SmogonEnv cqo0r - WARNING - Received 'bigerror' message: ['', 'bigerror', "You will auto-tie if the battle doesn't end in 30 turns (on turn 1000)."]
(MultiAgentEnvRunner pid=15752) 2025-11-20 20:16:39,299 - SmogonEnv asud0 - WARNING - Received 'bigerror' message: ['', 'bigerror', "You will auto-tie if the battle doesn't end in 20 turns (on turn 1000)."]
(MultiAgentEnvRunner pid=15752) 2025-11-20 20:16:39,300 - SmogonEnv cqo0r - WARNING - Received 'bigerror' message: ['', 'bigerror', "You will auto-tie if the battle doesn't end in 20 turns (on turn 1000)."]
(MultiAgentEnvRunner pid=15752) 2025-11-20 20:16:39,368 - SmogonEnv asud0 - WARNING - Received 'bigerror' message: ['', 'bigerror', "You will auto-tie if the battle doesn't

(MultiAgentEnvRunner pid=15752) {'SmogonEnv cqo0r': np.int32(3), 'SmogonEnv asud0': np.int32(1)}
(MultiAgentEnvRunner pid=15752) {'SmogonEnv cqo0r': np.int32(1), 'SmogonEnv asud0': np.int32(3)}
(MultiAgentEnvRunner pid=15752) {'SmogonEnv cqo0r': np.int32(3), 'SmogonEnv asud0': np.int32(1)}
(MultiAgentEnvRunner pid=15752) {'SmogonEnv cqo0r': np.int32(1), 'SmogonEnv asud0': np.int32(3)}
(MultiAgentEnvRunner pid=15752) {'SmogonEnv cqo0r': np.int32(3), 'SmogonEnv asud0': np.int32(1)}
(MultiAgentEnvRunner pid=15752) {'SmogonEnv cqo0r': np.int32(1), 'SmogonEnv asud0': np.int32(3)}
(MultiAgentEnvRunner pid=15752) {'SmogonEnv cqo0r': np.int32(3), 'SmogonEnv asud0': np.int32(1)}
(MultiAgentEnvRunner pid=15752) {'SmogonEnv cqo0r': np.int32(1), 'SmogonEnv asud0': np.int32(3)}
(MultiAgentEnvRunner pid=15752) {'SmogonEnv cqo0r': np.int32(3), 'SmogonEnv asud0': np.int32(1)}
(MultiAgentEnvRunner pid=15752) {'SmogonEnv cqo0r': np.int32(1), 'SmogonEnv asud0': np.int32(3)}
(MultiAgentEnvRunner pid=15752

(MultiAgentEnvRunner pid=15752) 2025-11-20 20:16:39,503 - SmogonEnv asud0 - PS_ERROR - Error message received: |error|[Invalid choice] Can't switch: You can't switch to an active Pokémon
(MultiAgentEnvRunner pid=15752) 2025-11-20 20:16:39,508 - SmogonEnv asud0 - PS_ERROR - Error message received: |error|[Invalid choice] Can't switch: You can't switch to an active Pokémon
(MultiAgentEnvRunner pid=15752) 2025-11-20 20:16:39,514 - SmogonEnv asud0 - PS_ERROR - Error message received: |error|[Invalid choice] Can't switch: You can't switch to an active Pokémon
(MultiAgentEnvRunner pid=15752) 2025-11-20 20:16:39,519 - SmogonEnv asud0 - PS_ERROR - Error message received: |error|[Invalid choice] Can't switch: You can't switch to an active Pokémon
(MultiAgentEnvRunner pid=15752) 2025-11-20 20:16:39,526 - SmogonEnv asud0 - PS_ERROR - Error message received: |error|[Invalid choice] Can't switch: You can't switch to an active Pokémon
(MultiAgentEnvRunner pid=15752) 2025-11-20 20:16:39,532 - SmogonE

(MultiAgentEnvRunner pid=15752) {'SmogonEnv cqo0r': np.int32(1), 'SmogonEnv asud0': np.int32(1)}
(MultiAgentEnvRunner pid=15752) {'SmogonEnv cqo0r': np.int32(3), 'SmogonEnv asud0': np.int32(1)}
(MultiAgentEnvRunner pid=15752) {'SmogonEnv cqo0r': np.int32(3), 'SmogonEnv asud0': np.int32(1)}
(MultiAgentEnvRunner pid=15752) {'SmogonEnv cqo0r': np.int32(3), 'SmogonEnv asud0': np.int32(1)}
(MultiAgentEnvRunner pid=15752) {'SmogonEnv cqo0r': np.int32(3), 'SmogonEnv asud0': np.int32(1)}
(MultiAgentEnvRunner pid=15752) {'SmogonEnv cqo0r': np.int32(3), 'SmogonEnv asud0': np.int32(1)}
(MultiAgentEnvRunner pid=15752) {'SmogonEnv cqo0r': np.int32(3), 'SmogonEnv asud0': np.int32(1)}
(MultiAgentEnvRunner pid=15752) {'SmogonEnv cqo0r': np.int32(3), 'SmogonEnv asud0': np.int32(1)}
(MultiAgentEnvRunner pid=15752) {'SmogonEnv cqo0r': np.int32(3), 'SmogonEnv asud0': np.int32(1)}
(MultiAgentEnvRunner pid=15752) {'SmogonEnv cqo0r': np.int32(3), 'SmogonEnv asud0': np.int32(1)}
(MultiAgentEnvRunner pid=15752

In [21]:
algo.train()

{'timers': {'training_iteration': 21.283222517000002,
  'restore_env_runners': 5.0825000000500605e-05,
  'training_step': 21.282802359,
  'env_runner_sampling_timer': 7.46033105,
  'learner_update_timer': 13.812221821999996,
  'synch_weights': 0.009639338999999934,
  'synch_env_connectors': 0.003638899999998557},
 'env_runners': {'episode_duration_sec_mean': 6.990999450000007,
  'module_episode_returns_mean': {'p1': 0.0},
  'module_to_env_connector': {'timers': {'connectors': {'get_actions': 0.00015850686882760127,
     'remove_single_ts_time_rank_from_batch': 2.6542928229704932e-06,
     'normalize_and_clip_actions': 5.989913308852836e-05,
     'listify_data_for_vector_env': 1.7002588531711332e-05,
     'tensor_to_numpy': 9.467722777406036e-05,
     'un_batch_to_individual_items': 5.0053295925726804e-05,
     'module_to_agent_unmapping': 6.1235888539429385e-06}},
   'connector_pipeline_timer': 0.0005076885612765258},
  'timers': {'connectors': {'agent_to_module_mapping': 8.30000000107

In [18]:
x = []

In [19]:
x == []

True

In [20]:
np.random.choice(np.where(x==1)[0])

ValueError: 'a' cannot be empty unless no samples are taken